# Convert raw to dsToolbox format

The `ds-Toolbox` wants input in the form of a specific csv-file pr charge or discharge pr temperature.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

import cellpy
from cellpy.utils import plotutils

cellpy.__version__

In [ ]:
cur_dir = Path(".")
raw_data_path = cur_dir / "raw" / "combined_protocol_results_realistic.bdf.csv"
assert raw_data_path.exists()

In [ ]:
c = cellpy.get(
    raw_data_path,
    instrument="batmo_bdf",
    cycle_mode="full_cell",
    mass=1.0,
    nominal_capacity=120.0,
    nom_cap_specifics="absolute",
    refuse_copying=True,
)

In [ ]:
c.data.raw.head()

In [ ]:
# We have a custom column that we want to also include in the summary (pr. cycle statistics) to find the cycles we want to export
c.add_to_summary("Maximum Cell Temperature / degC");

In [ ]:
plotutils.summary_plot(c.from_cycle(2), hover_columns=["Maximum Cell Temperature / degC"], formation_cycles=None)

In [ ]:
plotutils.summary_plot(c, y="capacities_absolute_with_rate", interactive=True, filters={"rate": (0.49, 0.51)}, hover_columns=["Maximum Cell Temperature / degC"], formation_cycles=None)

In [ ]:
# Helper function that we will need later when exporting (resetting step numbers, selecting step numbers, reversing the current)

from functools import partial

def tweak_something(df: pd.DataFrame, steps=None) -> pd.DataFrame:
    if steps is not None:
        df = df.query("step_index in @steps")
    df["step_index"] = (
    df.groupby("cycle_index")["step_index"]
      .transform(lambda s: pd.factorize(s)[0] + 1)
      .astype("int64")
)
    df.current = -df.current
    return df

## Export the first RPT

### First cycle (15 deg C)

In [ ]:
# We have to split into charge and discharge. Lets look at the data and find which ones.
plotutils.cycle_info_plot(c, cycle=2)

In [ ]:
# if we want to change from standard bdf units, we need to provide a CellpyUnits object with wanted units.

from cellpy.parameters.internal_settings import CellpyUnits
bdf_units = CellpyUnits(time="hrs")

In [ ]:
# Where to save the files

outdir = cur_dir / "out"
outdir.mkdir(exist_ok=True)

In [ ]:
c.to_bdf(outdir / "BattMo_OCV_P15_S3.csv", cycles=[2], extras=True, preprocess_fn=partial(tweak_something, steps=[6, 7, 8]), bdf_units=bdf_units)

In [ ]:
c.to_bdf(outdir / "BattMo_OCV_P15_S1.csv", cycles=[2], extras=True, preprocess_fn=partial(tweak_something, steps=[8, 9, 10]), bdf_units=bdf_units)

### Create the files for 25, 35, and 45 deg C

## Export the last RPT